# Load CSV

In [189]:
import pandas as pd

BASELINE_PATH = 'router no person/csi_data_20260313_122627.csv'   # ← đổi path
FILE_PATH = 'Router/1 nguoi ngoi/csi_data_20260313_124024.csv'  # ← đổi path
df = pd.read_csv(FILE_PATH)
df.head()

,timestamp,type,id,mac,rssi,rate,noise_floor,fft_gain,agc_gain,channel,local_timestamp,sig_len,rx_state,len,first_word,data
0,1.773380e+09,CSI_DATA,987588,98:4a:6b:31:4a:10,-58,11,-98,-34,54,1,-1728458458,83,0,256,0,"[0,0,0,0,0,0,0,0,19,50,11,61,0,58,-15,50,-30,5..."
1,1.773380e+09,CSI_DATA,987589,98:4a:6b:31:4a:10,-54,11,-98,16,37,1,-1728458071,83,0,256,0,"[0,0,0,0,0,0,0,0,77,-12,90,0,97,12,71,58,64,64..."
2,1.773380e+09,CSI_DATA,987590,98:4a:6b:31:4a:10,-57,11,-98,-38,55,1,-1728437355,83,0,256,0,"[0,0,0,0,0,0,0,0,19,-46,38,-38,42,-34,54,-15,5..."
3,1.773380e+09,CSI_DATA,987591,98:4a:6b:31:4a:10,-57,11,-98,-36,54,1,-1728427379,83,0,256,0,"[0,0,0,0,0,0,0,0,-36,40,-45,32,-57,16,-53,0,-6..."
4,1.773380e+09,CSI_DATA,987592,98:4a:6b:31:4a:10,-53,11,-98,14,37,1,-1728416359,83,0,256,0,"[0,0,0,0,0,0,0,0,-34,82,-55,68,-61,48,-82,27,-..."


In [190]:
# Tính FS thực từ timestamp
ts = pd.to_numeric(df['timestamp'])
FS_actual = 1.0 / ts.diff().dropna().median()
print(f"FS thực: {FS_actual:.1f} Hz")

# Dùng FS_actual thay FS=100 trong toàn bộ pipeline
FS = FS_actual


FS thực: 116.1 Hz


# Helper Functions

Định nghĩa các hàm xử lý dùng chung: parse I/Q, unwrap phase, Hampel, Savitzky-Golay, Butterworth.

In [191]:
import ast
import numpy as np
from scipy.signal import butter, sosfiltfilt, detrend, savgol_filter
from hampel_filter import hampel

# ── Config ───────────────────────────────────────────────────────────────
FS      = 100.0   # sampling rate (Hz)
LOWCUT  = 0.1     # bandpass low  (Hz)
HIGHCUT = 0.5     # bandpass high (Hz)
ORDER   = 4

# Subcarrier pilot / DC indices cần loại bỏ
_DROP = {
     64: list(range(0, 6))  + list(range(59, 64))  + [32],
    128: list(range(0, 6))  + list(range(122, 128)) + [63, 64, 65],
    256: list(range(0, 11)) + list(range(245, 256)) + [127, 128, 129],
}

def _parse_row(value, n_sub):
    """Parse chuỗi/list I/Q → (i, q) sau khi loại pilot/DC."""
    if isinstance(value, str):
        value = ast.literal_eval(value)
    arr = np.asarray(value, dtype=float)
    i, q = arr[0::2], arr[1::2]
    drop = _DROP.get(n_sub)
    if drop:
        mask = np.ones(n_sub, dtype=bool)
        mask[drop] = False
        i, q = i[mask], q[mask]
    return i, q

def _unwrap_sanitize_row(row_phase):
    """Loại 2π jump và timing offset theo chiều subcarrier."""
    u = np.unwrap(row_phase)
    k = np.arange(u.size)
    a, b = np.polyfit(k, u, 1)
    return u - (a * k + b)

def _hampel_1d(arr, window_size=20, n_sigma=3.0):
    """Hampel filter 1D — thay outlier bằng median cửa sổ."""
    arr = np.asarray(arr, dtype=float)
    if arr.size == 0:
        return arr
    outlier_indices = hampel(arr, window_size=window_size, n=n_sigma,
                             parallel=False, return_indices=True)
    if isinstance(outlier_indices, tuple):
        outlier_indices = outlier_indices[0]
    outlier_indices = np.asarray(outlier_indices, dtype=int)
    if outlier_indices.size == 0:
        return arr.copy()
    filtered = arr.copy()
    for idx in outlier_indices:
        start  = max(0, idx - window_size)
        end    = min(arr.size, idx + window_size + 1)
        window = np.concatenate([arr[start:idx], arr[idx + 1:end]])
        filtered[idx] = np.median(window)
    return filtered

def _savgol_1d(arr, window_length=31, polyorder=3):
    """Savitzky-Golay filter 1D."""
    arr = np.asarray(arr, dtype=float)
    if arr.size < 3:
        return arr
    wl = min(window_length, arr.size)
    if wl % 2 == 0:
        wl -= 1
    if wl < 3:
        return arr
    return savgol_filter(arr, window_length=wl, polyorder=min(polyorder, wl - 1))

_sos     = butter(ORDER, [LOWCUT, HIGHCUT], btype='band', fs=FS, output='sos')
MIN_LEN  = ORDER * 3 + 1

def _butter_1d(arr):
    """Butterworth bandpass filter 1D với detrend."""
    arr = np.asarray(arr, dtype=float)
    if arr.size < MIN_LEN:
        return arr
    return sosfiltfilt(_sos, detrend(arr, type='linear'))


# Pipeline Function

Gộp toàn bộ bước xử lý vào một hàm `run_pipeline(df)` để dùng lại cho cả data chính và baseline.

In [192]:
def run_pipeline(df):
    """
    Chạy toàn bộ pipeline CSI trên một DataFrame:
      parse I/Q → unwrap & sanitize → Hampel → Savitzky-Golay → Butterworth
    Trả về: amp_bw, phase_bw  (shape: n_rows × n_subcarriers)
    """
    # 1. Parse I/Q → amp & phase matrices
    amps, phases = [], []
    for _, row in df.iterrows():
        n_sub = int(row['len']) // 2
        i, q  = _parse_row(row['data'], n_sub)
        amps.append(np.hypot(i, q))
        phases.append(np.arctan2(q, i))
    amp_mat   = np.stack(amps)
    phase_mat = np.stack(phases)

    # 2. Phase unwrap & sanitize (spatial → temporal)
    phase_sp  = np.apply_along_axis(_unwrap_sanitize_row, axis=1, arr=phase_mat)
    phase_uw  = np.unwrap(phase_sp, axis=0)
    from scipy.signal import detrend
    phase_uw = detrend(phase_uw, axis=0, type='linear')

    # 3. Hampel filter (temporal, per subcarrier)
    amp_h   = np.apply_along_axis(_hampel_1d,  axis=0, arr=amp_mat)
    phase_h = np.apply_along_axis(_hampel_1d,  axis=0, arr=phase_uw)

    # # 4. Savitzky-Golay (temporal)
    amp_sg   = np.apply_along_axis(_savgol_1d, axis=0, arr=amp_h)
    phase_sg = np.apply_along_axis(_savgol_1d, axis=0, arr=phase_h)

    # 5. Butterworth bandpass (temporal)
    amp_bw   = np.apply_along_axis(_butter_1d, axis=0, arr=amp_sg)
    phase_bw = np.apply_along_axis(_butter_1d, axis=0, arr=phase_sg)
    # amp_bw   = np.apply_along_axis(_butter_1d, axis=0, arr=amp_h)
    # phase_bw = np.apply_along_axis(_butter_1d, axis=0, arr=phase_h)

    # skip 500 sample đầu để tránh transient response của filter
    amp_bw   = amp_bw[500:]
    phase_bw = phase_bw[500:]

    #     # Tìm subcarrier nhạy nhất với tín hiệu thở
    # from scipy.signal import welch
    # # import numpy as np

    # scores = []
    # for sub in range(amp_mat.shape[1]):
    #     sig = amp_bw[:, sub]  # sau butterworth
    #     freqs, psd = welch(sig, fs=FS, nperseg=int(FS * 8))
        
    #     breath_mask = (freqs >= 0.1) & (freqs <= 0.5)
    #     noise_mask  = (freqs >  0.5) & (freqs <= 2.0)
        
    #     peak_power  = psd[breath_mask].max()
    #     noise_floor = np.median(psd[noise_mask]) + 1e-10
    #     snr_db      = 10 * np.log10(peak_power / noise_floor)
    #     scores.append(snr_db)

    # scores = np.array(scores)
    # best_subs = np.argsort(scores)[::-1][:5]  # top 5 subcarrier

    # print("Top 5 subcarrier nhạy nhất:")
    # for sub in best_subs:
    #     print(f"  sub {sub:3d} — SNR {scores[sub]:.1f} dB")

    # # Plot top 3
    # fig = go.Figure()
    # for sub in best_subs[:3]:
    #     fig.add_trace(go.Scatter(y=amp_bw[:, sub], mode='lines',
    #                             name=f'sub {sub} ({scores[sub]:.1f} dB)'))
    # fig.update_layout(title='Top 3 subcarrier — amp sau Butterworth', height=400)
    # fig.show()

    

    return amp_bw, phase_bw


# ── Chạy pipeline trên data chính ────────────────────────────────────────
amp_bw, phase_bw = run_pipeline(df)
print(f"Data — amp_bw  : {amp_bw.shape}")
print(f"Data — phase_bw: {phase_bw.shape}")


Data — amp_bw  : (3500, 113)
Data — phase_bw: (3500, 113)


# Baseline Calibration

Load file CSV baseline (đo khi phòng trống), chạy qua cùng pipeline,
rồi **trừ mean của baseline** khỏi data để loại multipath tĩnh — chỉ giữ lại
thành phần biến động do chuyển động/hơi thở.

**Đổi `BASELINE_PATH`** thành đường dẫn file baseline của bạn.

In [193]:

# 1. Load & chạy pipeline trên baseline
df_base = pd.read_csv(BASELINE_PATH)
amp_bw_base, phase_bw_base = run_pipeline(df_base)
print(f'Baseline — amp_bw  : {amp_bw_base.shape}')
print(f'Data     — amp_bw  : {amp_bw.shape}')

# 2. Kiểm tra số subcarrier khớp nhau
assert amp_bw.shape[1] == amp_bw_base.shape[1], (
    f'Số subcarrier không khớp: data={amp_bw.shape[1]}, baseline={amp_bw_base.shape[1]}'
)

# 3. Align độ dài: lấy min(n_data, n_baseline) samples
#    Nếu baseline ngắn hơn data → tile (lặp lại) baseline cho đủ dài
n_data = amp_bw.shape[0]
n_base = amp_bw_base.shape[0]

if n_base >= n_data:
    # Baseline dài hơn hoặc bằng → dùng n_data sample đầu
    amp_base_aligned   = amp_bw_base[:n_data]
    phase_base_aligned = phase_bw_base[:n_data]
    print(f'Baseline dài hơn data → cắt {n_data} samples đầu')
else:
    # Baseline ngắn hơn → tile lặp lại rồi cắt
    reps = int(np.ceil(n_data / n_base))
    amp_base_aligned   = np.tile(amp_bw_base,   (reps, 1))[:n_data]
    phase_base_aligned = np.tile(phase_bw_base, (reps, 1))[:n_data]
    print(f'Baseline ngắn hơn data → tile x{reps}, cắt {n_data} samples')

# 4. Trừ baseline THEO THỜI GIAN — triệt tiêu cả dao động tĩnh lẫn multipath
#    Nếu data == baseline file → amp_calibrated ≈ 0 (noise sàn)
amp_calibrated   = amp_bw   - amp_base_aligned
phase_calibrated = phase_bw - phase_base_aligned

print(f'\nSau calibration (sample-wise subtraction):')
print(f'  amp_calibrated   std: {amp_calibrated.std():.6f}  (nếu ≈ 0 → đã triệt tiêu tốt)')
print(f'  phase_calibrated std: {phase_calibrated.std():.6f}')


Baseline — amp_bw  : (3500, 113)
Data     — amp_bw  : (3500, 113)
Baseline dài hơn data → cắt 3500 samples đầu

Sau calibration (sample-wise subtraction):
  amp_calibrated   std: 6.902981  (nếu ≈ 0 → đã triệt tiêu tốt)
  phase_calibrated std: 7.829515


# PCA

PCA chạy trên tín hiệu **đã trừ baseline** để trích đặc trưng.

In [194]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def _fit_pca(matrix, name, n_components=5):
    X_scaled = StandardScaler().fit_transform(matrix)
    pca = PCA(n_components=n_components, svd_solver='randomized', random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    print(f"\n── {name} ──")
    for i, (e, ec) in enumerate(zip(pca.explained_variance_ratio_,
                                     np.cumsum(pca.explained_variance_ratio_))):
        print(f"  PC{i+1}: {e*100:5.2f}%  cumulative: {ec*100:6.2f}%")
    return X_pca, pca

amp_pca,   pca_amp   = _fit_pca(amp_calibrated,   'Amplitude PCA (calibrated)')
phase_pca, pca_phase = _fit_pca(phase_calibrated, 'Phase PCA (calibrated)')



── Amplitude PCA (calibrated) ──
  PC1: 67.71%  cumulative:  67.71%
  PC2: 21.35%  cumulative:  89.07%
  PC3:  1.79%  cumulative:  90.86%
  PC4:  1.30%  cumulative:  92.16%
  PC5:  0.99%  cumulative:  93.15%

── Phase PCA (calibrated) ──
  PC1: 37.28%  cumulative:  37.28%
  PC2: 16.86%  cumulative:  54.15%
  PC3: 11.07%  cumulative:  65.22%
  PC4:  6.04%  cumulative:  71.26%
  PC5:  4.48%  cumulative:  75.74%


# Visualize PCA

In [195]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

x = np.arange(amp_pca.shape[0])
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Amplitude PCA (calibrated)', 'Phase PCA (calibrated)'))

for i in range(amp_pca.shape[1]):
    e = pca_amp.explained_variance_ratio_[i]
    fig.add_trace(go.Scatter(x=x, y=amp_pca[:, i], mode='lines',
                             name=f'amp PC{i+1} ({e*100:.1f}%)'), row=1, col=1)

for i in range(phase_pca.shape[1]):
    e = pca_phase.explained_variance_ratio_[i]
    fig.add_trace(go.Scatter(x=x, y=phase_pca[:, i], mode='lines',
                             name=f'phase PC{i+1} ({e*100:.1f}%)'), row=2, col=1)

fig.update_layout(height=700, title='PCA Components (calibrated)',
                  xaxis2_title='row', legend_title='component')
fig.show()


# Doppler Shift

Dùng **Phase PC0** của tín hiệu đã calibrate để tính Doppler velocity và spectrogram.

In [196]:
# ── STFT — energy dải thở theo thời gian ─────────────────────────────────
from scipy.signal import stft
import plotly.graph_objects as go
import numpy as np

# Dùng Phase PC0 sau baseline calibration
phase_signal = amp_pca[:, 0]

# ── 1. STFT ───────────────────────────────────────────────────────────────
# nperseg=4s → frequency resolution = 1/4 = 0.25 Hz
# noverlap=90% → time resolution mịn
nperseg  = int(FS * 4)
noverlap = int(nperseg * 0.9)

f, t, Zxx = stft(phase_signal, fs=FS, nperseg=nperseg,
                 noverlap=noverlap, window='hann')

# Chỉ giữ dải thở 0.1–0.5 Hz
mask     = (f >= 0.1) & (f <= 0.5)
f_breath = f[mask]
Z_breath = np.abs(Zxx[mask, :]) ** 2   # energy = |STFT|²

# ── 2. Plot spectrogram energy ────────────────────────────────────────────
fig = go.Figure(go.Heatmap(
    x=t,
    y=f_breath * 60,          # Hz → bpm cho dễ đọc
    z=10 * np.log10(Z_breath + 1e-10),
    colorscale='Jet',
    colorbar=dict(title='dB'),
))
fig.update_layout(
    title='STFT Energy — dải thở (0.1–0.5 Hz)',
    xaxis_title='Thời gian (s)',
    yaxis_title='Nhịp thở (bpm)',
    height=400,
)
fig.show()

# ── 3. Energy tổng theo thời gian (để thấy có người / không người) ────────
energy_over_time = Z_breath.sum(axis=0)   # sum energy toàn dải thở tại mỗi frame

fig2 = go.Figure(go.Scatter(
    x=t, y=energy_over_time,
    mode='lines', line=dict(width=1.5)
))
fig2.update_layout(
    title='Tổng energy dải thở theo thời gian',
    xaxis_title='Thời gian (s)',
    yaxis_title='Energy',
    height=300,
)
fig2.show()

In [204]:
# ── Respiratory Rate Extraction (theo paper Burimas et al. 2024) ──────────
from scipy.ndimage import gaussian_filter1d
from scipy.signal import butter, sosfiltfilt, find_peaks
from scipy.interpolate import interp1d
import numpy as np
import plotly.graph_objects as go

from scipy.signal import welch
    # import numpy as np

amps, phases = [], []
for _, row in df.iterrows():
    n_sub = int(row['len']) // 2
    i, q  = _parse_row(row['data'], n_sub)
    amps.append(np.hypot(i, q))
    phases.append(np.arctan2(q, i))
amp_mat   = np.stack(amps)
phase_mat = np.stack(phases)

scores = []
for sub in range(amp_mat.shape[1]):
    sig = amp_bw[:, sub]  # sau butterworth
    freqs, psd = welch(sig, fs=FS, nperseg=int(FS * 8))
    
    breath_mask = (freqs >= 0.1) & (freqs <= 0.5)
    noise_mask  = (freqs >  0.5) & (freqs <= 2.0)
    
    peak_power  = psd[breath_mask].max()
    noise_floor = np.median(psd[noise_mask]) + 1e-10
    snr_db      = 10 * np.log10(peak_power / noise_floor)
    scores.append(snr_db)

scores = np.array(scores)
best_subs = np.argsort(scores)[::-1][:5]  # top 5 subcarrier


# ── Config (theo paper) ───────────────────────────────────────────────────
SUB_IDX    = best_subs[0]   # dùng subcarrier nhạy nhất
FS_TARGET  = 60.0           # resample về 60 Hz (theo paper)
GAUSSIAN_SIGMA = 15         # γ — paper dùng 15 cho phòng nhỏ, 35 cho phòng lớn
BUTTER_ORDER   = 2          # δ
BUTTER_CUTOFF  = 1         # ε Hz — lowpass, KHÔNG phải bandpass
WINDOW_SEC     = 30         # cửa sổ đếm peak
Z_THRESHOLD    = 5          # τ — ngưỡng Z-score loại noise

# ── 1. Lấy raw amplitude 1 subcarrier ────────────────────────────────────
ts_raw  = pd.to_numeric(df['timestamp']).values
amp_raw = amp_mat[:, SUB_IDX].astype(float)

# ── 2. Hampel filter (đã có sẵn từ pipeline) ─────────────────────────────
amp_h = _hampel_1d(amp_raw)

# ── 3. Gaussian filter ────────────────────────────────────────────────────
amp_g = gaussian_filter1d(amp_h, sigma=GAUSSIAN_SIGMA)

# ── 4. Linear interpolation → resample về FS_TARGET Hz ───────────────────
t_uniform = np.arange(ts_raw[0], ts_raw[-1], 1.0 / FS_TARGET)
amp_interp = interp1d(ts_raw, amp_g, kind='linear',
                      bounds_error=False, fill_value='extrapolate')(t_uniform)

# Thử các cutoff khác nhau để tìm sóng thở rõ nhất
fig = go.Figure()
for cutoff in [0.5, 1.0, 2.0, 5.0]:
    sos = butter(2, cutoff, btype='low', fs=FS_TARGET, output='sos')
    # Detrend trước để bỏ drift
    amp_detrend = amp_interp - np.polyval(np.polyfit(np.arange(len(amp_interp)), amp_interp, 1), np.arange(len(amp_interp)))
    amp_filtered = sosfiltfilt(sos, amp_detrend)
    fig.add_trace(go.Scatter(x=t_sec[:len(amp_filtered)], y=amp_filtered,
                             mode='lines', name=f'cutoff={cutoff}Hz'))

fig.update_layout(title='So sánh cutoff — sau detrend', height=400)
fig.show()

# ── 7. Đếm peak trong từng cửa sổ 30s → bpm ─────────────────────────────
# min_distance: không ai thở > 30 bpm → peak cách nhau ít nhất 60/30 * FS_TARGET = 120 samples
MIN_DIST = int(FS_TARGET * 60 / 30)   # = 120 samples (theo paper ζ=120)

n_samples = len(amp_lp)
win_size  = int(WINDOW_SEC * FS_TARGET)
step_size = int(5 * FS_TARGET)         # slide mỗi 5s

bpm_list, t_list, zscore_list = [], [], []

for start in range(0, n_samples - win_size, step_size):
    window = amp_lp[start:start + win_size]
    
    # Z-score anomaly detection (theo paper)
    z = (window - window.mean()) / (window.std() + 1e-10)
    z_range = z.max() - z.min()
    zscore_list.append(z_range)
    t_list.append(t_sec[start + win_size // 2])
    
    if z_range > Z_THRESHOLD:
        bpm_list.append(None)   # noise window → bỏ
        continue
    
    peaks, _ = find_peaks(window, distance=MIN_DIST)
    bpm = len(peaks) * (60.0 / WINDOW_SEC)
    bpm_list.append(bpm)

# ── 8. Plot bpm theo thời gian ────────────────────────────────────────────
bpm_valid = [b for b in bpm_list if b is not None]
t_valid   = [t for b, t in zip(bpm_list, t_list) if b is not None]
t_noise   = [t for b, t in zip(bpm_list, t_list) if b is None]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_valid, y=bpm_valid, mode='lines+markers',
                          name='RR (bpm)', line=dict(width=2)))
if t_noise:
    fig2.add_trace(go.Scatter(x=t_noise, y=[None]*len(t_noise),
                              mode='markers', marker=dict(color='red', size=8),
                              name='noise window'))
fig2.add_hrect(y0=12, y1=20, fillcolor='green', opacity=0.08, line_width=0)
fig2.update_layout(title='Respiratory Rate ước tính',
                   xaxis_title='Thời gian (s)', yaxis_title='bpm', height=350)
fig2.show()

# ── 9. Tóm tắt ───────────────────────────────────────────────────────────
if bpm_valid:
    print(f"RR trung bình : {np.mean(bpm_valid):.1f} bpm")
    print(f"RR min/max    : {np.min(bpm_valid):.1f} / {np.max(bpm_valid):.1f} bpm")
    print(f"Window noise  : {bpm_list.count(None)}/{len(bpm_list)}")

/tmp/ipykernel_792586/1052577125.py:30: RuntimeWarning: divide by zero encountered in log10
  snr_db      = 10 * np.log10(peak_power / noise_floor)


RR trung bình : 18.4 bpm
RR min/max    : 18.0 / 20.0 bpm
Window noise  : 0/5


# Test

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.signal import butter, sosfiltfilt, find_peaks
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import plotly.graph_objects as go
import plotly.figure_factory as ff

# ── Config ────────────────────────────────────────────────────────────────
DATA_ROOT   = 'Router'          # ← đổi thành path gốc chứa các thư mục class
FS_TARGET   = 60.0              # resample về 60 Hz
GAUSSIAN_SIGMA = 15
BUTTER_ORDER   = 2
BUTTER_CUTOFF  = 1.0            # lowpass 1 Hz — giữ dải thở
MIN_DIST    = int(FS_TARGET * 60 / 30)   # min distance giữa peaks

# Label mapping — presence detection (binary) hoặc multi-class
LABEL_MAP = {
    '1 nguoi nam bat quat' : 'co_nguoi',
    '1 nguoi ngoi'         : 'co_nguoi',
    '1 nguoi ngoi bat quat': 'co_nguoi',
    '2 nguoi ngoi bat quat': 'co_nguoi',
    'nhieu nguoi bat quat' : 'co_nguoi',
    'khong nguoi'          : 'khong_nguoi',
    'khong nguoi bat quat' : 'khong_nguoi',
}
# Hoặc dùng multi-class — comment dòng trên, bỏ comment dưới:
# LABEL_MAP = {k: k for k in os.listdir(DATA_ROOT)}

# ── Helper functions (giữ nguyên từ pipeline) ─────────────────────────────
_DROP = {
    64 : list(range(0, 6))  + list(range(59, 64))  + [32],
    128: list(range(0, 6))  + list(range(122, 128)) + [63, 64, 65],
    256: list(range(0, 11)) + list(range(245, 256)) + [127, 128, 129],
}

def get_timestamps(df_f):
    """Lấy timestamp theo thứ tự ưu tiên: timestamp → local_timestamp → index-based"""
    for col in ['timestamp', 'local_timestamp']:
        if col in df_f.columns:
            ts = pd.to_numeric(df_f[col], errors='coerce')
            if ts.notna().sum() > len(df_f) * 0.9:   # ít nhất 90% hợp lệ
                return ts.interpolate().values         # fill NaN nếu có vài chỗ thiếu
    
    # Không có timestamp → tạo từ index với FS giả định 100Hz
    print(f"    Không có timestamp — dùng index-based 100Hz")
    return np.arange(len(df_f)) / 100.0

def _parse_row(value, n_sub):
    if isinstance(value, str):
        value = ast.literal_eval(value)
    arr = np.asarray(value, dtype=float)
    i, q = arr[0::2], arr[1::2]
    drop = _DROP.get(n_sub)
    if drop:
        mask = np.ones(n_sub, dtype=bool)
        mask[drop] = False
        i, q = i[mask], q[mask]
    return i, q

def _hampel_1d(arr, window_size=20, n_sigma=3.0):
    """Vectorized hampel — nhanh hơn loop ~100x"""
    arr = arr.astype(float)
    n = len(arr)
    result = arr.copy()
    
    # Dùng pandas rolling median — vectorized
    s = pd.Series(arr)
    rolling_median = s.rolling(window=2*window_size+1, center=True, min_periods=1).median()
    rolling_mad    = (s - rolling_median).abs().rolling(
                        window=2*window_size+1, center=True, min_periods=1).median()
    
    threshold = n_sigma * 1.4826 * rolling_mad
    outliers  = np.abs(arr - rolling_median.values) > threshold.values
    result[outliers] = rolling_median.values[outliers]
    return result

def process_signal(sig, ts_raw):
    """Xử lý 1 subcarrier — giải phóng memory ngay sau dùng"""
    sig_h  = _hampel_1d(sig)
    sig_g  = gaussian_filter1d(sig_h, sigma=GAUSSIAN_SIGMA)
    
    t_uniform = np.arange(ts_raw[0], ts_raw[-1], 1.0 / FS_TARGET)
    sig_i = interp1d(ts_raw, sig_g, kind='linear',
                     bounds_error=False, fill_value='extrapolate')(t_uniform)
    
    sos    = butter(BUTTER_ORDER, BUTTER_CUTOFF, btype='low', fs=FS_TARGET, output='sos')
    sig_lp = sosfiltfilt(sos, sig_i)
    
    x = np.arange(len(sig_lp))
    sig_lp -= np.polyval(np.polyfit(x, sig_lp, 1), x)
    return sig_lp, t_uniform

# ── Feature extraction từ 1 signal ───────────────────────────────────────
def extract_features(sig):
    """Trích features từ signal đã processed."""
    peaks, _   = find_peaks(sig, distance=MIN_DIST)
    troughs, _ = find_peaks(-sig, distance=MIN_DIST)

    # Tần số dao động
    if len(peaks) >= 2:
        peak_intervals = np.diff(peaks) / FS_TARGET   # giây
        mean_period    = peak_intervals.mean()
        std_period     = peak_intervals.std()
        dom_freq       = 1.0 / mean_period if mean_period > 0 else 0
    else:
        mean_period = std_period = dom_freq = 0

    # Biên độ
    if len(peaks) > 0 and len(troughs) > 0:
        amp_mean = (sig[peaks].mean() - sig[troughs].mean()) / 2
        amp_std  = sig[peaks].std()
    else:
        amp_mean = amp_std = 0

    # Energy trong dải thở (0.1–0.5 Hz) qua FFT
    fft_mag  = np.abs(np.fft.rfft(sig))
    fft_freq = np.fft.rfftfreq(len(sig), d=1.0/FS_TARGET)
    breath_mask = (fft_freq >= 0.1) & (fft_freq <= 0.5)
    noise_mask  = (fft_freq >  0.5) & (fft_freq <= 2.0)
    energy_breath = fft_mag[breath_mask].mean()
    energy_noise  = fft_mag[noise_mask].mean()  + 1e-10
    snr = energy_breath / energy_noise

    return {
        'variance'     : sig.var(),
        'std'          : sig.std(),
        'peak_count'   : len(peaks),
        'dom_freq_hz'  : dom_freq,
        'mean_period'  : mean_period,
        'std_period'   : std_period,
        'amp_mean'     : amp_mean,
        'amp_std'      : amp_std,
        'energy_breath': energy_breath,
        'energy_noise' : energy_noise,
        'snr'          : snr,
        'kurtosis'     : float(pd.Series(sig).kurtosis()),
        'skewness'     : float(pd.Series(sig).skew()),
    }

# ── Load toàn bộ dataset ──────────────────────────────────────────────────
records = []

for class_dir in os.listdir(DATA_ROOT):
    class_path = os.path.join(DATA_ROOT, class_dir)
    if not os.path.isdir(class_path):
        continue
    label = LABEL_MAP.get(class_dir)
    if label is None:
        print(f"  Bỏ qua: {class_dir} (không có trong LABEL_MAP)")
        continue

    csv_files = [f for f in os.listdir(class_path) if f.endswith('.csv')]
    print(f"  {class_dir} ({label}): {len(csv_files)} files")

    for fname in csv_files:
        try:
            df_f = pd.read_csv(fpath)
            ts   = get_timestamps(df_f)

            # Parse amp — chỉ giữ best subcarrier, bỏ matrix
            amps = []
            for _, row in df_f.iterrows():
                n_sub = int(row['len']) // 2
                i, q  = _parse_row(row['data'], n_sub)
                amps.append(np.hypot(i, q))
            amp_mat = np.stack(amps)

            best_sub = np.argmax(amp_mat.var(axis=0))
            sig_raw  = amp_mat[:, best_sub].astype(float)

            # Giải phóng ngay
            del amp_mat, amps, df_f

            sig_proc, _ = process_signal(sig_raw, ts)
            del sig_raw

            feats = extract_features(sig_proc)
            del sig_proc

            feats['label'] = label
            feats['file']  = fname
            feats['class'] = class_dir
            records.append(feats)

        except Exception as e:
            print(f"    Lỗi {fname}: {e}")

df_feat = pd.DataFrame(records)
print(f"\nTổng: {len(df_feat)} files, {df_feat['label'].value_counts().to_dict()}")
df_feat.head()

IndentationError: expected an indented block after 'for' statement on line 162 (4137278924.py, line 163)

In [ ]:
# ── Train & Evaluate ──────────────────────────────────────────────────────
feature_cols = ['variance','std','peak_count','dom_freq_hz','mean_period',
                'std_period','amp_mean','amp_std','energy_breath',
                'energy_noise','snr','kurtosis','skewness']

X = df_feat[feature_cols].fillna(0).values
y = LabelEncoder().fit_transform(df_feat['label'])
label_names = LabelEncoder().fit(df_feat['label']).classes_

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Cross-validation 5-fold
clf = RandomForestClassifier(n_estimators=100, random_state=42)
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring='accuracy')

print(f"CV Accuracy: {scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%")

# Train full + confusion matrix
clf.fit(X_scaled, y)
y_pred = clf.predict(X_scaled)
print("\n", classification_report(y, y_pred, target_names=label_names))

# Plot confusion matrix
cm = confusion_matrix(y, y_pred)
fig = ff.create_annotated_heatmap(cm, x=list(label_names), y=list(label_names),
                                  colorscale='Blues')
fig.update_layout(title='Confusion Matrix', height=400,
                  xaxis_title='Predicted', yaxis_title='Actual')
fig.show()

# Feature importance
importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=True)
fig2 = go.Figure(go.Bar(x=importances.values, y=importances.index, orientation='h'))
fig2.update_layout(title='Feature Importance', height=400)
fig2.show()

NameError: name 'df_feat' is not defined